In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import ast
import os

In [2]:
SAVE_PATH = "../results/figures/data_analysis"
SURVIVAL_TABLE_PATH = "../data/interim/matched_clinical_slides.csv"

In [3]:
# 폰트 및 해상도 설정
plt.rcdefaults()
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['figure.dpi'] = 600

# 저장할 폴더 생성
save_dir = SAVE_PATH
os.makedirs(save_dir, exist_ok=True)

# 색상 팔레트
DEAD_COLOR = '#E74C3C'
ALIVE_COLOR = '#2ECC71'
ACCENT = '#3498DB'

# ==========================================
# 0. 데이터 로드 및 매핑 (수정된 부분)
# ==========================================
df = pd.read_csv(SURVIVAL_TABLE_PATH)

# 이 데이터셋은 이미 환자(submitter_id) 1명당 1줄로 요약되어 있습니다.
# 컬럼명 매핑
df['os_time'] = df['time']
df['dead_or_alive'] = df['vital_status']
df['num_slides'] = df['n_slides']


# ==========================================
# 1. 파이 차트 - 생사 분포 (Vital Status Distribution)
# ==========================================
fig, ax = plt.subplots(figsize=(7, 5))
counts = df['dead_or_alive'].value_counts()
wedges, texts, autotexts = ax.pie(
    counts.values, labels=counts.index, colors=[ALIVE_COLOR, DEAD_COLOR],
    autopct=lambda pct: f'{pct:.1f}%\n({int(round(pct/100.*sum(counts.values)))})',
    startangle=90, textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}, explode=(0.03, 0.03)
)
for autotext in autotexts:
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')
ax.set_title(f'Patient Vital Status Distribution\n(TCGA-LUAD, N={len(df)})', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{save_dir}/1_vital_status_pie.png', bbox_inches='tight')
plt.close()


# ==========================================
# 2. 박스플롯 - 생존 기간 비교 (Survival Time Boxplot)
# ==========================================
fig, ax = plt.subplots(figsize=(7, 5))
dead_times = df[df['dead_or_alive'] == 'Dead']['os_time'].dropna()
alive_times = df[df['dead_or_alive'] == 'Alive']['os_time'].dropna()

bp = ax.boxplot(
    [dead_times, alive_times],
    tick_labels=['Dead', 'Alive'],
    patch_artist=True, widths=0.5,
    boxprops=dict(linewidth=1.5),
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    flierprops=dict(marker='o', markersize=4, alpha=0.5)
)
bp['boxes'][0].set_facecolor(DEAD_COLOR)
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(ALIVE_COLOR)
bp['boxes'][1].set_alpha(0.7)

for i, (data, label) in enumerate([(dead_times, 'Dead'), (alive_times, 'Alive')]):
    median = data.median()
    mean = data.mean()
    ax.text(i+1, data.max() + 100, f'Median: {median:.0f}d\nMean: {mean:.0f}d\nN={len(data)}',
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray', alpha=0.8))

ax.set_ylabel('Days', fontsize=12, fontweight='bold')
ax.set_title('Survival Time Distribution by Vital Status\n(TCGA-LUAD)', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{save_dir}/2_survival_boxplot.png', bbox_inches='tight')
plt.close()


# ==========================================
# 3. 끊어진 막대 차트 (Broken Y-axis) - 슬라이드 분포
# ==========================================
slide_counts = df['num_slides'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5), gridspec_kw={'height_ratios': [1, 2]})
fig.subplots_adjust(hspace=0.1)

x_pos = np.arange(len(slide_counts))
bars1 = ax1.bar(x_pos, slide_counts.values, color=ACCENT, edgecolor='white', linewidth=1.5)
bars2 = ax2.bar(x_pos, slide_counts.values, color=ACCENT, edgecolor='white', linewidth=1.5)

ax1.set_ylim(400, 460) # 상단 축 범위
ax2.set_ylim(0, 12)    # 하단 축 범위

# 가운데 끊어지는 선 그리기
ax1.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax1.tick_params(labeltop=False, bottom=False)
ax2.xaxis.tick_bottom()

d = .015 
kwargs = dict(transform=ax1.transAxes, color='black', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        
ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  
kwargs.update(transform=ax2.transAxes)  
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  
ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

# 텍스트 추가
for bar, val in zip(bars1, slide_counts.values):
    if val > 100:  
        ax1.text(bar.get_x() + bar.get_width()/2, val + 2, str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar, val in zip(bars2, slide_counts.values):
    if val <= 100:  
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.3, str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(slide_counts.index.astype(str))
ax2.set_xlabel('Number of Slides per Patient', fontsize=12, fontweight='bold')
fig.text(0.04, 0.5, 'Number of Patients', va='center', rotation='vertical', fontsize=12, fontweight='bold')
ax1.set_title(f'Slide Count Distribution per Patient\n(TCGA-LUAD, N={len(df)})', fontsize=14, fontweight='bold', pad=15)
ax1.grid(axis='y', alpha=0.3)
ax2.grid(axis='y', alpha=0.3)

plt.savefig(f'{save_dir}/3_slides_per_patient_bar.png', bbox_inches='tight')
plt.close()


# ==========================================
# 4. 누적 막대 차트 - 기간별 생사 분포
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))
bins = [0, 365, 1095, 1825, 99999]
labels_bin = ['< 1 year', '1-3 years', '3-5 years', '5+ years']
df['os_bin'] = pd.cut(df['os_time'], bins=bins, labels=labels_bin, right=True)

cross = pd.crosstab(df['os_bin'], df['dead_or_alive'])
if 'Dead' not in cross.columns: cross['Dead'] = 0
if 'Alive' not in cross.columns: cross['Alive'] = 0
cross = cross[['Dead', 'Alive']]

cross.plot(kind='bar', stacked=True, color=[DEAD_COLOR, ALIVE_COLOR], ax=ax,
           edgecolor='white', linewidth=1.5, width=0.6)

for i, (idx, row) in enumerate(cross.iterrows()):
    cum = 0
    for col in ['Dead', 'Alive']:
        val = row[col]
        if val > 0:
            ax.text(i, cum + val/2, str(int(val)), ha='center', va='center', fontsize=10, fontweight='bold', color='white')
        cum += val
    ax.text(i, cum + 3, str(int(cum)), ha='center', va='bottom', fontsize=10, fontweight='bold', color='black')

ax.set_xlabel('Survival Period', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Patients', fontsize=12, fontweight='bold')
ax.set_title(f'Patient Distribution by Survival Period\n(TCGA-LUAD, N={len(df)})', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(labels_bin, rotation=0)
ax.legend(title='Status', fontsize=10, title_fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{save_dir}/4_survival_period_stacked_bar.png', bbox_inches='tight')
plt.close()


# ==========================================
# 5. Kaplan-Meier 생존 곡선
# ==========================================
fig, ax = plt.subplots(figsize=(8, 5))
km_df = df[['os_time', 'dead_or_alive']].dropna().copy()
km_df['event'] = (km_df['dead_or_alive'] == 'Dead').astype(int)
km_df = km_df.sort_values('os_time')

n = len(km_df)
times = [0]
surv_prob = [1.0]
at_risk = n

for t, event in zip(km_df['os_time'], km_df['event']):
    if event == 1:
        surv_prob.append(surv_prob[-1] * (1 - 1/at_risk))
        times.append(t)
    at_risk -= 1

censor_times = km_df[km_df['event'] == 0]['os_time'].values

ax.step(times, surv_prob, where='post', color=ACCENT, linewidth=2.5, label='Overall Survival')
ax.fill_between(times, surv_prob, step='post', alpha=0.1, color=ACCENT)

for ct in censor_times:
    idx = np.searchsorted(times, ct, side='right') - 1
    if 0 <= idx < len(surv_prob):
        ax.plot(ct, surv_prob[idx], '|', color='gray', markersize=6, alpha=0.5)

median_surv = None
for i in range(len(surv_prob)):
    if surv_prob[i] <= 0.5:
        median_surv = times[i]
        break

if median_surv:
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=median_surv, color='gray', linestyle='--', alpha=0.5)
    ax.text(median_surv + 50, 0.52, f'Median OS: {median_surv:.0f} days\n({median_surv/365:.1f} years)',
            fontsize=10, fontweight='bold', color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray', alpha=0.8))

ax.set_xlabel('Time (Days)', fontsize=12, fontweight='bold')
ax.set_ylabel('Survival Probability', fontsize=12, fontweight='bold')
ax.set_title(f'Kaplan-Meier Overall Survival Curve\n(TCGA-LUAD, N={len(df)})', fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(-0.02, 1.05)
ax.set_xlim(-50, max(times) + 200)
ax.legend(loc='lower left', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{save_dir}/5_kaplan_meier_curve.png', bbox_inches='tight')
plt.close()


# ==========================================
# 6. 데이터셋 요약표 (Table 1)
# ==========================================
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

alive_n = (df['dead_or_alive'] == 'Alive').sum()
dead_n = (df['dead_or_alive'] == 'Dead').sum()
total_n = len(df)
total_slides = df['num_slides'].sum()
median_os = df['os_time'].median()
median_followup = df[df['dead_or_alive'] == 'Alive']['os_time'].median()
median_death = df[df['dead_or_alive'] == 'Dead']['os_time'].median()

table_data = [
    ['Total Patients', f'{total_n}'],
    ['Total Slides', f'{total_slides}'],
    ['Dead (Event)', f'{dead_n} ({dead_n/total_n*100:.1f}%)'],
    ['Alive (Censored)', f'{alive_n} ({alive_n/total_n*100:.1f}%)'],
    ['Median OS Time (Dead)', f'{median_death:.0f} days ({median_death/365:.1f} years)'],
    ['Median Follow-up (Alive)', f'{median_followup:.0f} days ({median_followup/365:.1f} years)'],
    ['Median Overall OS Time', f'{median_os:.0f} days ({median_os/365:.1f} years)'],
]

table = ax.table(cellText=table_data, colLabels=['Metric', 'Value'],
                 loc='center', cellLoc='center', colWidths=[0.45, 0.45])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#ECF0F1')
    cell.set_edgecolor('#BDC3C7')

ax.set_title('Dataset Summary (Table 1)\nTCGA-LUAD Cohort', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{save_dir}/6_summary_table.png', bbox_inches='tight')
plt.close()

print(f'All 6 charts generated using {SURVIVAL_TABLE_PATH} and saved to {SAVE_PATH}!')


All 6 charts generated using ../data/interim/matched_clinical_slides.csv and saved to ../results/figures/data_analysis!
